# Goose Man: ทดสอบประสิทธิภาพตัวจัดเส้นทาง

รันบน Colab เพื่อไม่ให้กินแรมเครื่อง: **Runtime → Run all** ใช้เวลาประมาณ 5 นาที

| ส่วน | วัดอะไร |
|---|---|
| A | `planRoute` ใช้เวลาเท่าไร เมื่อคนหิ้วถือ 1-6 ออเดอร์ |
| B | `suggestAddOns` ใช้เวลาเท่าไร เมื่อมีงานเปิดรอ 10-200 งาน |
| C | รับหลายงานในรอบเดียว ประหยัดเวลาเท่าไร เทียบกับส่งทีละงาน |
| D | เทียบคุณภาพเส้นทางกับ OR-Tools (ตัวแก้โจทย์ใน `rout_hack/03`) บนโจทย์ชุดเดียวกัน |

แคมปัสในการทดสอบเป็นแบบจำลองเดียวกับ `rout_hack/03` (พื้นที่ 800 ม., ร้านรวมกลุ่มรอบโรงอาหาร, ตึก 1-12 ชั้น)
ไม่ใช่แผนที่ มจธ. จริง ตัวเลขความเร็วจึงใช้ได้ แต่ตัวเลขเวลาเดินจริงต้องรอแผนที่

สร้างโดย `bench/routing/make_notebook.py` อย่าแก้โค้ดในโน้ตบุ๊กนี้ตรงๆ ให้แก้ใน repo แล้วสร้างใหม่

## 1. เตรียมเครื่อง

In [ ]:
%%bash
# Node ≥ 22.6 runs TypeScript directly (type stripping)
if ! node -e "const [a,b]=process.versions.node.split('.').map(Number);process.exit(a>22||(a===22&&b>=6)?0:1)" 2>/dev/null; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null
  apt-get install -y nodejs > /dev/null
fi
node -v

In [ ]:
!pip install -q ortools

In [ ]:
!mkdir -p /content/goose/bench /content/goose/bench/routing /content/goose/frontend/src/lib/routing

## 2. โค้ดจาก repo

In [ ]:
%%writefile /content/goose/frontend/package.json
{ "private": true, "type": "module" }


In [ ]:
%%writefile /content/goose/frontend/src/lib/routing/cost.ts
// Walking-time model, ported from rout_hack/03_campus_food_delivery/core/costmodel.py:
//
//   time(A → B) = ground distance / speed + FLOOR_PENALTY × (floor(B) − 1)
//
// Ground distance is the straight line times a detour factor, because
// campus paths are not straight. Replace `travelSeconds` with a lookup on a
// real path graph once one exists; the planner only calls it through
// `TravelFn`, so nothing else has to change.
import type { Place } from './places';

export type TravelFn = (from: Place, to: Place, toFloor?: number) => number;

export interface CostConfig {
	/** Walking speed on campus, m/s */
	speedMps: number;
	/** Straight line → real path length */
	detourFactor: number;
	/** Lift/stairs time per floor above the ground floor, seconds */
	floorPenaltyS: number;
	/** Multiplies every ground leg: 1 = normal, ~1.4 = heavy rain */
	slowdown: number;
}

export const DEFAULT_COST: CostConfig = { speedMps: 1.3, detourFactor: 1.3, floorPenaltyS: 25, slowdown: 1 };

const EARTH_RADIUS_M = 6_371_000;
const rad = (deg: number) => (deg * Math.PI) / 180;

export function distanceMeters(a: Place, b: Place): number {
	const dLat = rad(b.lat - a.lat);
	const dLng = rad(b.lng - a.lng);
	const h = Math.sin(dLat / 2) ** 2 + Math.cos(rad(a.lat)) * Math.cos(rad(b.lat)) * Math.sin(dLng / 2) ** 2;
	return 2 * EARTH_RADIUS_M * Math.asin(Math.sqrt(h));
}

export function travelFn(cfg: Partial<CostConfig> = {}): TravelFn {
	const c = { ...DEFAULT_COST, ...cfg };
	return (from, to, toFloor = 1) => {
		const ground = from.id === to.id ? 0 : (distanceMeters(from, to) * c.detourFactor * c.slowdown) / c.speedMps;
		return ground + Math.max(0, toFloor - 1) * c.floorPenaltyS;
	};
}


In [ ]:
%%writefile /content/goose/frontend/src/lib/routing/planner.ts
// Route planning for one rider, ported from the OR-Tools PDPTW in
// rout_hack/03_campus_food_delivery/core/pdp_solver.py.
//
// Goose Man riders pick their own jobs, so there is no central dispatcher
// here: the planner sequences the jobs one rider already holds, and scores
// open jobs they could add. A rider holds a handful of orders at most, so
// every valid stop order is searched exhaustively and the answer is exact.
//
// Same hard constraints as the Python solver:
//   1. an order's pickup comes before its delivery
//   2. at most `capacity` orders per outing
//   3. no pickup before the food is ready (the rider waits)
//   4. no delivery after the order's deadline
// Once a rider has started delivering, the outing is closed: no new pickups.
import type { Place } from './places';
import type { TravelFn } from './cost';

export interface RouteOrder {
	id: string;
	pickup: Place;
	dropoff: Place;
	dropoffFloor?: number;
	/** Seconds from now until the food is ready. Default: ready now */
	readyAt?: number;
	/** Seconds from now by which it must be delivered. Default: no deadline */
	deadline?: number;
	/** Already in the rider's hands: no pickup stop */
	pickedUp?: boolean;
}

export interface RouteStop {
	kind: 'pickup' | 'delivery';
	orderId: string;
	place: Place;
	/** Seconds from now */
	arriveAt: number;
	/** Seconds spent waiting for the food */
	wait: number;
}

export interface Route {
	stops: RouteStop[];
	/** Walking time only */
	travelSeconds: number;
	/** When the last delivery lands, seconds from now */
	finishSeconds: number;
}

/** Exhaustive search grows fast; riders never carry this many */
export const MAX_PLAN_ORDERS = 6;

export function planRoute(start: Place, orders: RouteOrder[], travel: TravelFn, capacity: number): Route | null {
	if (orders.length === 0) return { stops: [], travelSeconds: 0, finishSeconds: 0 };
	if (orders.length > capacity || orders.length > MAX_PLAN_ORDERS) return null;

	const picked = orders.map((o) => !!o.pickedUp);
	const delivered = orders.map(() => false);
	const path: RouteStop[] = [];
	let best: Route | null = null;

	const better = (travelS: number, finishS: number) =>
		!best || travelS < best.travelSeconds - 1e-9 || (Math.abs(travelS - best.travelSeconds) <= 1e-9 && finishS < best.finishSeconds);

	function search(at: Place, now: number, travelS: number, left: number) {
		if (best && travelS > best.travelSeconds + 1e-9) return;
		if (left === 0) {
			if (better(travelS, now)) best = { stops: [...path], travelSeconds: travelS, finishSeconds: now };
			return;
		}
		orders.forEach((o, i) => {
			if (delivered[i]) return;
			if (!picked[i]) {
				const leg = travel(at, o.pickup);
				const arrive = now + leg;
				const depart = Math.max(arrive, o.readyAt ?? 0);
				if (depart > (o.deadline ?? Infinity)) return; // can't even collect it in time
				picked[i] = true;
				path.push({ kind: 'pickup', orderId: o.id, place: o.pickup, arriveAt: arrive, wait: depart - arrive });
				search(o.pickup, depart, travelS + leg, left);
				path.pop();
				picked[i] = false;
			} else {
				const leg = travel(at, o.dropoff, o.dropoffFloor);
				const arrive = now + leg;
				if (arrive > (o.deadline ?? Infinity)) return;
				delivered[i] = true;
				path.push({ kind: 'delivery', orderId: o.id, place: o.dropoff, arriveAt: arrive, wait: 0 });
				search(o.dropoff, arrive, travelS + leg, left - 1);
				path.pop();
				delivered[i] = false;
			}
		});
	}

	search(start, 0, 0, orders.length);
	return best;
}

export interface AddOnSuggestion {
	order: RouteOrder;
	route: Route;
	/** How much later the rider finishes by taking this job too */
	extraSeconds: number;
	/** Largest delay this causes to a job the rider already holds */
	maxDelaySeconds: number;
}

export interface SuggestOptions {
	capacity: number;
	/** Hide jobs that add more than this. Default 5 minutes */
	maxExtraSeconds?: number;
}

/** Open jobs worth taking on the way, cheapest first */
export function suggestAddOns(
	start: Place,
	current: RouteOrder[],
	candidates: RouteOrder[],
	travel: TravelFn,
	{ capacity, maxExtraSeconds = 300 }: SuggestOptions
): AddOnSuggestion[] {
	// One outing per round: once food is in hand, no going back for more
	if (current.length >= capacity || current.some((o) => o.pickedUp)) return [];
	const base = planRoute(start, current, travel, capacity);
	if (!base) return [];
	const baseArrival = deliveryTimes(base);

	const suggestions: AddOnSuggestion[] = [];
	for (const order of candidates) {
		if (order.pickedUp || current.some((o) => o.id === order.id)) continue;
		const route = planRoute(start, [...current, order], travel, capacity);
		if (!route) continue;
		const extraSeconds = route.finishSeconds - base.finishSeconds;
		if (extraSeconds > maxExtraSeconds) continue;
		const arrival = deliveryTimes(route);
		const maxDelaySeconds = Math.max(0, ...current.map((o) => arrival.get(o.id)! - baseArrival.get(o.id)!));
		suggestions.push({ order, route, extraSeconds, maxDelaySeconds });
	}
	return suggestions.sort((a, b) => a.extraSeconds - b.extraSeconds || a.maxDelaySeconds - b.maxDelaySeconds);
}

function deliveryTimes(route: Route): Map<string, number> {
	return new Map(route.stops.filter((s) => s.kind === 'delivery').map((s) => [s.orderId, s.arriveAt]));
}


In [ ]:
%%writefile /content/goose/bench/package.json
{
	"private": true,
	"type": "module"
}


In [ ]:
%%writefile /content/goose/bench/routing/bench.ts
// Route planner benchmark. Plain Node ≥ 22.6, no dependencies:
//   node --experimental-strip-types bench/routing/bench.ts [--quick]
//
// Campus generator mirrors rout_hack/03_campus_food_delivery/bench/generate_campus.py:
// an 800 m square, stalls clustered within 120 m of a food court, buildings
// scattered with 1-12 floors. Writes bench/routing/instances.json for
// ortools_compare.py.
import { writeFileSync } from 'node:fs';
import { performance } from 'node:perf_hooks';
import { travelFn } from '../../frontend/src/lib/routing/cost.ts';
import { planRoute, suggestAddOns } from '../../frontend/src/lib/routing/planner.ts';
import type { Place } from '../../frontend/src/lib/routing/places.ts';
import type { RouteOrder } from '../../frontend/src/lib/routing/planner.ts';

const QUICK = process.argv.includes('--quick');
const travel = travelFn();
const CAPACITY = 4;
const SLA_S = 2400; // 40 min, same default as the Python PoC

// --- seeded campus ---------------------------------------------------------
function rng(seed: number) {
	return () => {
		seed = (seed + 0x6d2b79f5) | 0;
		let t = Math.imul(seed ^ (seed >>> 15), 1 | seed);
		t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t;
		return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
	};
}
const LAT0 = 13.65;
const M_PER_DEG_LAT = 111_320;
const M_PER_DEG_LNG = 111_320 * Math.cos((LAT0 * Math.PI) / 180);
const point = (id: string, x: number, y: number): Place => ({ id, lat: LAT0 + y / M_PER_DEG_LAT, lng: 100.49 + x / M_PER_DEG_LNG });

interface Campus {
	stores: Place[];
	buildings: { place: Place; floors: number }[];
}

function campus(r: () => number): Campus {
	const stores = Array.from({ length: 25 }, (_, i) => {
		const ang = r() * 2 * Math.PI;
		const rad = r() * 120;
		return point(`s${i}`, 400 + rad * Math.cos(ang), 400 + rad * Math.sin(ang));
	});
	const floorsChoice = [1, 3, 5, 8, 12];
	const buildings = Array.from({ length: 10 }, (_, i) => ({
		place: point(`b${i}`, r() * 800, r() * 800),
		floors: floorsChoice[Math.floor(r() * floorsChoice.length)]
	}));
	return { stores, buildings };
}

function makeOrders(r: () => number, c: Campus, n: number, prefix = 'o'): RouteOrder[] {
	return Array.from({ length: n }, (_, i) => {
		const b = c.buildings[Math.floor(r() * c.buildings.length)];
		return {
			id: `${prefix}${i}`,
			pickup: c.stores[Math.floor(r() * c.stores.length)],
			dropoff: b.place,
			dropoffFloor: 1 + Math.floor(r() * b.floors),
			readyAt: 180 + r() * 420, // prep 3-10 min, as in the PoC
			deadline: SLA_S
		};
	});
}

const start = (r: () => number) => point('start', r() * 800, r() * 800);

// --- helpers ---------------------------------------------------------------
function timed<T>(fn: () => T): [T, number] {
	const t0 = performance.now();
	const out = fn();
	return [out, performance.now() - t0];
}
const pct = (xs: number[], p: number) => {
	const s = [...xs].sort((a, b) => a - b);
	return s[Math.min(s.length - 1, Math.floor((p / 100) * s.length))] ?? NaN;
};
const mean = (xs: number[]) => xs.reduce((a, b) => a + b, 0) / xs.length;
const f = (x: number, d = 2) => (Number.isFinite(x) ? x.toFixed(d) : '-');
function table(rows: Record<string, string | number>[]) {
	console.table(rows);
}

// --- A: planRoute latency --------------------------------------------------
console.log(`\n=== A. planRoute latency (capacity ${Math.max(6, CAPACITY)}, SLA ${SLA_S / 60} min) ===`);
{
	const rows = [];
	for (let n = 1; n <= 6; n++) {
		const r = rng(100 + n);
		const runs = QUICK ? 20 : n >= 6 ? 100 : 300;
		const ms: number[] = [];
		let feasible = 0;
		for (let k = 0; k < runs; k++) {
			const c = campus(r);
			const [route, t] = timed(() => planRoute(start(r), makeOrders(r, c, n), travel, 6));
			ms.push(t);
			if (route) feasible++;
		}
		rows.push({ orders: n, runs, 'median ms': f(pct(ms, 50), 3), 'p95 ms': f(pct(ms, 95), 3), 'max ms': f(Math.max(...ms), 3), feasible: `${feasible}/${runs}` });
	}
	table(rows);
}

// --- B: suggestAddOns latency ----------------------------------------------
console.log(`\n=== B. suggestAddOns latency (rider holds 2 jobs, capacity ${CAPACITY}, max +5 min) ===`);
{
	const rows = [];
	for (const m of [10, 50, 100, 200]) {
		const r = rng(200 + m);
		const runs = QUICK ? 5 : 50;
		const ms: number[] = [];
		const found: number[] = [];
		for (let k = 0; k < runs; k++) {
			const c = campus(r);
			const current = makeOrders(r, c, 2, 'mine');
			const open = makeOrders(r, c, m, 'open');
			const [out, t] = timed(() => suggestAddOns(start(r), current, open, travel, { capacity: CAPACITY }));
			ms.push(t);
			found.push(out.length);
		}
		rows.push({ 'open jobs': m, runs, 'median ms': f(pct(ms, 50)), 'p95 ms': f(pct(ms, 95)), 'max ms': f(Math.max(...ms)), 'avg suggestions': f(mean(found), 1) });
	}
	table(rows);
}

// --- C: batching vs one job per trip ---------------------------------------
console.log('\n=== C. Batched outing vs one job per trip (same rider, same jobs) ===');
{
	function oneAtATime(from: Place, orders: RouteOrder[]) {
		// Best order to do the jobs one by one: try every permutation (n ≤ 4)
		let best = { finish: Infinity, avgDelivered: Infinity };
		const permute = (rest: RouteOrder[], done: RouteOrder[]) => {
			if (rest.length) return rest.forEach((o, i) => permute([...rest.slice(0, i), ...rest.slice(i + 1)], [...done, o]));
			let at = from;
			let now = 0;
			const delivered: number[] = [];
			for (const o of done) {
				now = Math.max(now + travel(at, o.pickup), o.readyAt ?? 0);
				now += travel(o.pickup, o.dropoff, o.dropoffFloor);
				if (now > (o.deadline ?? Infinity)) return;
				delivered.push(now);
				at = o.dropoff;
			}
			if (now < best.finish) best = { finish: now, avgDelivered: mean(delivered) };
		};
		permute(orders, []);
		return best;
	}

	const rows = [];
	for (let n = 2; n <= 4; n++) {
		const r = rng(300 + n);
		const runs = QUICK ? 20 : 300;
		const saveFinish: number[] = [];
		const avgBatched: number[] = [];
		const avgSingle: number[] = [];
		for (let k = 0; k < runs; k++) {
			const c = campus(r);
			const s = start(r);
			const orders = makeOrders(r, c, n);
			const batched = planRoute(s, orders, travel, CAPACITY);
			const single = oneAtATime(s, orders);
			if (!batched || !Number.isFinite(single.finish)) continue;
			saveFinish.push(1 - batched.finishSeconds / single.finish);
			avgBatched.push(mean(batched.stops.filter((x) => x.kind === 'delivery').map((x) => x.arriveAt)));
			avgSingle.push(single.avgDelivered);
		}
		rows.push({
			jobs: n,
			compared: saveFinish.length,
			'rider done sooner': `${f(mean(saveFinish) * 100, 1)}%`,
			'avg customer wait batched (min)': f(mean(avgBatched) / 60, 1),
			'avg customer wait 1-by-1 (min)': f(mean(avgSingle) / 60, 1)
		});
	}
	table(rows);
}

// --- D: export instances for OR-Tools --------------------------------------
{
	const instances = [];
	for (let n = 2; n <= 6; n++) {
		const r = rng(400 + n);
		const runs = QUICK ? 3 : 30;
		for (let k = 0; k < runs; k++) {
			const c = campus(r);
			const s = start(r);
			const orders = makeOrders(r, c, n);
			const [route, ms] = timed(() => planRoute(s, orders, travel, 6));
			// node 0 = start, then pickup/delivery pairs: 2i+1 = pickup of order i, 2i+2 = its delivery
			const nodes = [{ place: s, floor: 1 }, ...orders.flatMap((o) => [{ place: o.pickup, floor: 1 }, { place: o.dropoff, floor: o.dropoffFloor ?? 1 }])];
			instances.push({
				n,
				matrix: nodes.map((a) => nodes.map((b) => travel(a.place, b.place, b.floor))),
				ready: orders.map((o) => o.readyAt ?? 0),
				deadline: orders.map((o) => o.deadline ?? null),
				ours: route ? { travel: route.travelSeconds, finish: route.finishSeconds, ms } : { travel: null, finish: null, ms }
			});
		}
	}
	writeFileSync(new URL('./instances.json', import.meta.url), JSON.stringify(instances));
	console.log(`\nWrote ${instances.length} instances to bench/routing/instances.json for ortools_compare.py`);
}


In [ ]:
%%writefile /content/goose/bench/routing/ortools_compare.py
"""Compare the TypeScript planner with OR-Tools on the same instances.

Reads bench/routing/instances.json (written by bench.ts) and solves each one
as a single-rider PDPTW with the same constraints as
rout_hack/03_campus_food_delivery/core/pdp_solver.py: pickup before delivery
on the same route, capacity, no pickup before the food is ready, delivery
before the deadline. The route is open (the rider does not return), so a
zero-cost dummy end node is used.

    pip install ortools
    python bench/routing/ortools_compare.py [--time-limit 1.0]

The TypeScript planner searches every stop order, so OR-Tools should never
find a shorter walk. A row with "OR-Tools shorter" > 0 means a planner bug.
"""

import argparse
import json
import statistics
import time
from collections import defaultdict
from pathlib import Path

from ortools.constraint_solver import pywrapcp, routing_enums_pb2

SCALE = 100  # OR-Tools needs integers: work in 1/100 s


def solve(inst, time_limit_s):
    n = inst["n"]
    matrix = inst["matrix"]
    size = 2 * n + 1
    end = size  # dummy end node
    manager = pywrapcp.RoutingIndexManager(size + 1, 1, [0], [end])
    routing = pywrapcp.RoutingModel(manager)

    def transit(i_idx, j_idx):
        i, j = manager.IndexToNode(i_idx), manager.IndexToNode(j_idx)
        if j == end or i == end:
            return 0
        return int(round(matrix[i][j] * SCALE))

    cb = routing.RegisterTransitCallback(transit)
    routing.SetArcCostEvaluatorOfAllVehicles(cb)

    horizon = int((max(d for d in inst["deadline"] if d is not None) if any(inst["deadline"]) else 10_800) * SCALE) + 1
    routing.AddDimension(cb, horizon, horizon, True, "Time")
    time_dim = routing.GetDimensionOrDie("Time")

    def demand(i_idx):
        node = manager.IndexToNode(i_idx)
        if node == 0 or node == end:
            return 0
        return 1 if node % 2 == 1 else -1

    dcb = routing.RegisterUnaryTransitCallback(demand)
    routing.AddDimensionWithVehicleCapacity(dcb, 0, [n], True, "Capacity")

    for i in range(n):
        p = manager.NodeToIndex(2 * i + 1)
        d = manager.NodeToIndex(2 * i + 2)
        routing.AddPickupAndDelivery(p, d)
        routing.solver().Add(routing.VehicleVar(p) == routing.VehicleVar(d))
        routing.solver().Add(time_dim.CumulVar(p) <= time_dim.CumulVar(d))
        time_dim.CumulVar(p).SetMin(int(round(inst["ready"][i] * SCALE)))
        if inst["deadline"][i] is not None:
            time_dim.CumulVar(d).SetMax(int(inst["deadline"][i] * SCALE))

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.time_limit.FromMilliseconds(int(time_limit_s * 1000))

    t0 = time.perf_counter()
    sol = routing.SolveWithParameters(params)
    ms = (time.perf_counter() - t0) * 1000
    if sol is None:
        return None, ms

    # Recompute the walk in float seconds from the route, not from the rounded objective
    idx, walk = routing.Start(0), 0.0
    while not routing.IsEnd(idx):
        nxt = sol.Value(routing.NextVar(idx))
        i, j = manager.IndexToNode(idx), manager.IndexToNode(nxt)
        if j != end:
            walk += matrix[i][j]
        idx = nxt
    return walk, ms


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--time-limit", type=float, default=1.0, help="OR-Tools seconds per instance")
    ap.add_argument("--file", default=str(Path(__file__).with_name("instances.json")))
    args = ap.parse_args()

    instances = json.loads(Path(args.file).read_text())
    by_n = defaultdict(list)
    for inst in instances:
        ortools_walk, ortools_ms = solve(inst, args.time_limit)
        by_n[inst["n"]].append((inst["ours"], ortools_walk, ortools_ms))

    tol = 0.05  # seconds: rounding to 1/100 s per leg
    print(f"\nOR-Tools time limit: {args.time_limit}s per instance (GLS keeps searching until the limit)\n")
    header = f"{'orders':>6} {'runs':>5} {'both feasible':>13} {'only ours found':>15} {'only OR-Tools found':>19} {'OR-Tools shorter':>16} {'avg OR-Tools gap':>16} {'max gap':>8} {'ours median ms':>14} {'OR-Tools median ms':>18}"
    print(header)
    print("-" * len(header))
    for n in sorted(by_n):
        rows = by_n[n]
        gaps, both, only_ours, only_ortools, ortools_better = [], 0, 0, 0, 0
        for ours, walk, _ in rows:
            ours_walk = ours["travel"]
            if ours_walk is not None and walk is None:
                only_ours += 1  # OR-Tools ran out of time before finding a route
                continue
            if ours_walk is None and walk is not None:
                only_ortools += 1  # would be a planner bug
                continue
            if ours_walk is None:
                continue
            both += 1
            if walk < ours_walk - tol:
                ortools_better += 1
            gaps.append((walk - ours_walk) / ours_walk * 100 if ours_walk > 0 else 0.0)
        ours_ms = statistics.median(o["ms"] for o, _, _ in rows)
        or_ms = statistics.median(m for _, _, m in rows)
        avg_gap = f"{statistics.mean(gaps):.2f}%" if gaps else "-"
        max_gap = f"{max(gaps):.2f}%" if gaps else "-"
        print(f"{n:>6} {len(rows):>5} {both:>13} {only_ours:>15} {only_ortools:>19} {ortools_better:>16} {avg_gap:>16} {max_gap:>8} {ours_ms:>14.3f} {or_ms:>18.1f}")
    print("\ngap = how much longer OR-Tools walks than the TypeScript planner (0% = same route quality)")
    print("'only ours found' = OR-Tools ran out of time; 'only OR-Tools found' or 'OR-Tools shorter' > 0 = planner bug")


if __name__ == "__main__":
    main()


## 3. รันทดสอบ (A-C และส่งออกโจทย์ให้ D)

In [ ]:
%cd /content/goose
!node --experimental-strip-types --no-warnings bench/routing/bench.ts

## 4. เทียบกับ OR-Tools (D)

In [ ]:
!python bench/routing/ortools_compare.py --time-limit 1

## อ่านผลอย่างไร

- **A:** ชุดทดสอบนี้ตั้งให้คนหิ้วถือได้ไม่เกิน 4 ออเดอร์ จึงดูแถว 1-4 เป็นหลัก (แถว 5-6 มีไว้ดูว่าเริ่มช้าตรงไหน) ถ้าต่ำกว่า 16 ms ผู้ใช้จะไม่รู้สึกว่ารอ
- **B:** `avg suggestions` คือจำนวนงานที่ผ่านเกณฑ์ "เพิ่มเวลาไม่เกิน 5 นาที" ถ้าเยอะมาก หน้าจอควรแสดงแค่ 3 อันดับแรก
- **C:** `rider done sooner` = คนหิ้วเสร็จเร็วขึ้นกี่ % เมื่อรับรวมรอบเดียว ส่วน `avg customer wait` ใช้ดูว่าลูกค้าต้องรอนานขึ้นหรือเปล่า
- **D:** `OR-Tools shorter` และ `only OR-Tools found` ต้องเป็น 0 ทุกแถว ถ้าไม่ใช่ แปลว่าตัวจัดเส้นทางมีบั๊ก
  `only ours found` ไม่ใช่บั๊ก แปลว่า OR-Tools หาคำตอบไม่ทันเวลาที่ให้ (ลองเพิ่ม `--time-limit`)